<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

%pip -q install duckdb huggingface_hub
import duckdb, pandas as pd, numpy as np, os

con = duckdb.connect()
con.execute("PRAGMA threads=1")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


## 1. Ranked actions + reason codes

Rebuilding the client-grouped model from w05/w06 (same features, same split logic,
now confirmed reproducible with `threads=1`) and scoring the full dataset to produce
a ranked action queue — the deliverable a reviewer actually works from.

In [9]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev90,
           SUM(gsc_clicks)      AS clk_prev90,
           AVG(gsc_avg_position) AS pos_avg_prev90,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_prev90,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_active_prev90
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev90 >= 100
    ORDER BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1, 2
    ORDER BY client_hash_id, content_hash_id
""").df()

data = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')
data['is_declining_future'] = (data['imp_next30'] < 0.8 * (data['imp_prev90'] / 3)).astype(int)

feature_cols = ['imp_prev90', 'clk_prev90', 'pos_avg_prev90', 'ctr_prev90', 'days_active_prev90']
model_data = data.dropna(subset=feature_cols + ['is_declining_future'])
X, y, groups = model_data[feature_cols], model_data['is_declining_future'], model_data['client_hash_id']

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1).fit(X_tr, y_tr)
print(f"Model trained on {len(X_tr):,} rows, {groups.iloc[train_idx].nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model trained on 105,653 rows, 33 clients


In [10]:
model_data['risk_score'] = model.predict_proba(model_data[feature_cols])[:, 1]

def assign_reason_code(row):
    if row['risk_score'] >= 0.65:
        return 'model_high_decline_risk'
    elif row['pos_avg_prev90'] <= 20 and row['ctr_prev90'] == 0:
        return 'visible_zero_ctr'
    elif row['days_active_prev90'] < 10 and row['imp_prev90'] < 200:
        return 'low_data_new_page'
    else:
        return 'routine_monitor'

model_data['reason_code'] = model_data.apply(assign_reason_code, axis=1)

# Anchor 'review' to a realistic reviewer capacity, not an arbitrary score threshold.
# w01 established Precision@K is the right metric because reviewers see a limited list —
# a threshold that flags 34% of all pages isn't a queue, it's most of the dataset.
REVIEW_CAPACITY = 200  # e.g. a reviewer working through ~200 pages over a review cycle
WATCH_CAPACITY = 1000

queue = model_data.sort_values('risk_score', ascending=False).reset_index(drop=True)

def assign_action_by_rank(row_idx, reason_code):
    if reason_code == 'low_data_new_page':
        return 'monitor_only'
    elif row_idx < REVIEW_CAPACITY:
        return 'review'
    elif row_idx < WATCH_CAPACITY:
        return 'watch'
    else:
        return 'no_action'

queue['action'] = [assign_action_by_rank(i, rc) for i, rc in enumerate(queue['reason_code'])]

print(queue['action'].value_counts())
print("\nTop 10:")
queue.head(10)[['client_hash_id', 'content_hash_id', 'risk_score', 'reason_code', 'action']]

action
no_action       114319
watch              800
monitor_only       298
review             200
Name: count, dtype: int64

Top 10:


,client_hash_id,content_hash_id,risk_score,reason_code,action
0,client_3197e6291363b4db,content_bddc07c4ef754ede,1.000,model_high_decline_risk,review
1,client_73cda7b4e4f265ea,content_8d4373aaa2e74790,1.000,model_high_decline_risk,review
2,client_62f4a7e64f5e0096,content_f07a1e60e7f9b911,1.000,model_high_decline_risk,review
3,client_23a62021009f63c4,content_58e570c1603f2dd5,1.000,model_high_decline_risk,review
4,client_62f4a7e64f5e0096,content_c8f9121890cdde98,1.000,model_high_decline_risk,review
5,client_62f4a7e64f5e0096,content_88917279943615dc,1.000,model_high_decline_risk,review
6,client_62f4a7e64f5e0096,content_290d7142d9189ab8,1.000,model_high_decline_risk,review
7,client_62f4a7e64f5e0096,content_c851c3150024cc95,1.000,model_high_decline_risk,review
8,client_23a62021009f63c4,content_20a3acfd025de44b,0.995,model_high_decline_risk,review
9,client_73cda7b4e4f265ea,content_57159812e0f8ffe8,0.995,model_high_decline_risk,review


With actions anchored to reviewer capacity rather than a raw score threshold, the
queue splits into `review` (200, the top-ranked candidates a reviewer could
realistically work through in a cycle), `watch` (800, a secondary lighter-monitoring
tier), `monitor_only` (298 — new pages the model deliberately declines to score
confidently, per w04's finding that low-activity pages trend toward growth, not
decline), and `no_action` (114,319). This shape matches the "limited review
capacity" framing from w01 — a 200-page `review` bucket is something one person
could realistically work through, unlike the earlier fixed-score threshold that
swept in 39,064 pages regardless of what a reviewer could handle.

The top 10 by `risk_score` all score at or near 1.000, and span 4 distinct clients —
better spread than a single-client sweep, though `client_62f4a7e64f5e0096` appears
4 of the top 8 times, worth a note rather than treating the top 10 as fully
declustered. Given w05's finding that per-client calibration varies (some clients
over- or under-predicted), a client with several genuinely high-risk pages
clustering near the top isn't necessarily wrong — but it's exactly the kind of
pattern Section 2's calibration caveat exists to flag before a reviewer assumes
every top-10 page is an independent, equally-confident signal.

## 2. Intended use and limits

**Who uses this:** a content/SEO reviewer with limited weekly review capacity,
exactly the persona defined in w01 — someone deciding which of many candidate pages
to look at first, not someone looking for a fully automated decision.

**What it's valid for:** ranking pages by observed decline risk over a single
90-day-feature → 30-day-outcome snapshot (Dec 2025–Mar 2026 features, Apr 2026
outcome), verified in w06 to be free of the leaks caught in w03 and reproducible
across runs (w05's fix). Precision@50 was measured at 0.680 on a client-grouped
holdout — meaning roughly 34 of the top 50 flagged pages can be expected to show
real future decline, a 2.0x improvement over the tier-adjusted baseline rule (0.340).

**Where it stops being valid:**
- This is one snapshot in time. w06 confirmed the result is reproducible *within*
  this snapshot, but nothing here validates that the same lift holds on a different
  month or after real seasonal shifts — re-validation on a new window is required
  before trusting this queue on fresh data.
- w05's per-client calibration check found real over/under-prediction for specific
  clients (`client_3f0ce4d44fe94f3d`, `client_1a730cb2640a1abf` overpredicted;
  `client_9958f0a7ae1df715` underpredicted) — this queue's scores should not be
  read as equally well-calibrated across every client.
- This is decision-support, not causal: a `review` flag means "worth a reviewer's
  time," not "refreshing this page will cause it to recover" — no experiment has
  been run to support that stronger claim (per w01/w02's own caution).

## 3. Human review + the no-go list

**What a reviewer must check before acting on a flagged page:**
- Whether the page is genuinely new (`days_active_prev90` low) rather than
  declining — the `low_data_new_page` reason code flags this, but it's worth a
  manual glance, since w04's Signal B found low-activity pages actually trend
  toward *growth*, not decline, in this data.
- Whether a `visible_zero_ctr` flag reflects a real underperformance or a SERP
  feature (e.g. featured snippet) that suppresses clicks without indicating a real
  problem — w04 flagged this exact ambiguity for near-zero position values.
- Whether the page belongs to one of the miscalibrated clients named in Section 2,
  where the score may run systematically high or low.

**What should never be automated:**
- Publishing or unpublishing a page based on this queue alone.
- Any action framed as "this refresh will recover traffic" — the data supports
  ranking for review, not a causal refresh-then-recover claim.
- Treating `action = 'no_action'` as proof a page has no problem — it means the
  model didn't flag elevated risk in this snapshot, not that the page was
  independently verified healthy.

## 4. Monitoring / retrain triggers

**Signs the recommendations have gone stale:**
- A new month's data becomes available and Precision@50 on a fresh client-grouped
  holdout drops meaningfully below the 0.680 baseline established here — would
  indicate the pattern isn't holding on new data.
- The reviewed `review`-flagged pages, tracked over the following 30–60 days, show
  a lower real-world decline rate than the ~68% implied by Precision@50 — a
  practical, not just statistical, check.
- A shift in the per-client calibration gaps found in w05 — if previously
  well-calibrated clients start showing large over/under-prediction, that's a sign
  the model needs retraining on more recent data.
- Any change to how `fact_content_daily_performance` or `fact_content_query_90d`
  are structured upstream (column renames, new fixed-window quirks like the one
  caught in w03) — would silently break the feature pipeline without an explicit
  recheck.

**Suggested cadence:** re-validate Precision@50 on a fresh month before each
capstone-adjacent milestone, not on a fixed calendar schedule — since the dataset
itself (Section 2) only covers one snapshot, staleness is better caught by
re-measuring than by assuming a fixed shelf life.

## 5. Exports for the paper

Writing the ranked queue and a metrics summary to `work/outputs/` — these feed the
capstone paper directly.

In [11]:
import json

os.makedirs('work/outputs', exist_ok=True)

export_cols = ['client_hash_id', 'content_hash_id', 'imp_prev90', 'pos_avg_prev90',
               'ctr_prev90', 'days_active_prev90', 'risk_score', 'reason_code', 'action']
queue[export_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/action_playbook_queue.csv")

metrics_summary = {
    'model': 'RandomForestClassifier(n_estimators=200)',
    'split': 'client-grouped (GroupShuffleSplit, test_size=0.25, random_state=42)',
    'feature_window': '2025-12-31 to 2026-03-31',
    'label_window': '2026-04-01 to 2026-04-30',
    'precision_at_50_model': 0.680,
    'precision_at_50_baseline': 0.340,
    'lift': 2.0,
    'auc': 0.720,
    'action_counts': queue['action'].value_counts().to_dict(),
    'reproducibility_note': 'Requires PRAGMA threads=1 for deterministic DuckDB aggregation (see w05).'
}
with open('work/outputs/action_playbook_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print("Wrote work/outputs/action_playbook_metrics.json")

Wrote 115,617 rows to work/outputs/action_playbook_queue.csv
Wrote work/outputs/action_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.